In [7]:
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [8]:
# --- 1. Hyperparameters & Settings ---
DATA_DIR = "dataset"
CLASSES = ["normal", "fault"]
SAMPLE_RATE = 16000
N_MFCC = 13  # Keep it small for future embedded deployment
EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = 0.001

In [9]:
# --- 2. Feature Extraction ---
def extract_mfcc(file_path):
    """Loads a WAV file and extracts the mean MFCCs."""
    try:
        # Load audio
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        # Average the features over time to get a single 1D array per file
        mfccs_mean = np.mean(mfccs.T, axis=0)
        return mfccs_mean
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [10]:
# --- 3. Dataset Preparation ---
class PumpAudioDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

def load_data():
    X, y = [], []
    for label_idx, label_name in enumerate(CLASSES):
        folder_path = os.path.join(DATA_DIR, label_name)
        if not os.path.exists(folder_path):
            continue
            
        for file in os.listdir(folder_path):
            if file.endswith(".wav"):
                file_path = os.path.join(folder_path, file)
                features = extract_mfcc(file_path)
                if features is not None:
                    X.append(features)
                    y.append(label_idx)
                    
    return np.array(X), np.array(y)


In [11]:
# --- 4. The Neural Network ---
class PumpFaultDetector(nn.Module):
    def __init__(self, input_size):
        super(PumpFaultDetector, self).__init__()
        # A simple, lightweight feed-forward network
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2) # 2 Output classes: Normal (0), Fault (1)
        )

    def forward(self, x):
        return self.network(x)

In [12]:
# --- 5. Main Training Loop ---
print("Extracting features from audio files...")
X, y = load_data()

if len(X) == 0:
    print("No data found! Make sure your dataset folder is populated.")
    exit()

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = PumpAudioDataset(X_train, y_train)
test_dataset = PumpAudioDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initialize model, loss, and optimizer
model = PumpFaultDetector(input_size=N_MFCC)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"\nTraining Model on {len(X_train)} samples...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# Evaluate the model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

print(f"\nFinal Test Accuracy: {100 * correct / total:.2f}%")

# Save the weights for later use
torch.save(model.state_dict(), "pump_fault_model.pth")
print("Model saved to pump_fault_model.pth")

Extracting features from audio files...

Training Model on 11 samples...
Epoch 10/50 | Loss: 0.0071
Epoch 20/50 | Loss: 0.0006
Epoch 30/50 | Loss: 0.0027
Epoch 40/50 | Loss: 0.0078
Epoch 50/50 | Loss: 0.0005

Final Test Accuracy: 66.67%
Model saved to pump_fault_model.pth
